<a href="https://colab.research.google.com/github/spirosChv/neuro208/blob/main/practicals/Exercise2_BCM_rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# @title Make nicer plots -- Execute this cell
def mystyle():
  """
  Create custom plotting style.

  Returns
  -------
  my_style : dict
      Dictionary with matplotlib parameters.

  """
  # color pallette
  style = {
      # Use LaTeX to write all text
      "text.usetex": False,
      "font.family": "DejaVu Sans",
      "font.weight": "bold",
      # Use 16pt font in plots, to match 16pt font in document
      "axes.labelsize": 16,
      "axes.titlesize": 20,
      "font.size": 16,
      # Make the legend/label fonts a little smaller
      "legend.fontsize": 14,
      "xtick.labelsize": 14,
      "ytick.labelsize": 14,
      "axes.linewidth": 2.5,
      "lines.markersize": 10.0,
      "lines.linewidth": 2.5,
      "xtick.major.width": 2.2,
      "ytick.major.width": 2.2,
      "axes.labelweight": "bold",
      "axes.spines.right": False,
      "axes.spines.top": False
  }

  return style


plt.style.use("seaborn-colorblind")
plt.rcParams.update(mystyle())

## Bienenstock Cooper Munro (BCM) Rule

Here, we will implement the competition between two input patterns $\textbf{x}_1=(20,0)$ and $\textbf{x}_2=(0,20)$. At each timestep, one of the two patterns is presented to the neuron. The pattern is chosen randomly with a probability 0.5 of being pattern $\textbf{x}_1$ and 0.5 of being pattern $\textbf{x}_1$. The weights follow the BCM rule. The output of the neuron at each timestep is given by

\begin{equation}
y(t) = \textbf{w}^{\text{T}}\textbf{x}(t)
\end{equation}

where $\textbf{x}(t)$ is the input pattern at time $t$.

\begin{align}
\frac{dw}{dt} &= \eta \textbf{x} y (y-\theta) \\
\tau_\theta\frac{d\theta}{dt} &= -\theta + \frac{\langle y \rangle^2}{y_0}
\end{align}

where $\theta$ is the sliding threshold$. 

In [ ]:
# simulations parameters
N = 2  # number of input patterns
T = 10**4  # time of simulation
dt = 1  # simulation time step
steps = int(T/dt)  # number of steps
tvec = np.linspace(0, T, steps+1)

In [ ]:
eta_w = 10**(-6)  # learning rate for weights
ytarget = 10  # neuronal output target 
x = 20*np.eye(N)  # inputs have two patterns one (0-20) and two (20-0)
tau_theta = 50  # time constant for theta

In [ ]:
# Initialization
y = np.zeros((steps+1, ))  # neuronal output
w = 0.5*np.ones((N, steps+1))  # weights
theta = 5*np.ones((steps+1, ))  # sliding threshold theta

In [ ]:
# Simulation
for n in range(steps):
  p = np.round(np.random.randint(N))  # presentation of a pattern randomly
  y[n] = w[:, n].T @ x[:, p]  # compute the output

  dw = ((y[n]**2)/ytarget - theta[n])/tau_theta  # dtheta/dt calculation
  theta[n+1] = theta[n] + dt*dw  # update sliding theshold

  dw = eta_w*x[:, p]*y[n]*(y[n] - theta[n])  # dw/dt calculation
  w[:, n+1] = w[:,n] + dt*dw  # update of the weights
  w[:, n+1] = (w[:, n+1] > 0) * w[:, n+1]  # weigths can't be negative: hard bound at zero

In [ ]:
# Plot
plt.figure(figsize=(10, 8))
plt.subplot(3, 1, 1)
for i in range(N):
  plt.plot(tvec, w[i, :], label=f'w{i}')
plt.ylabel('w')
plt.legend()
plt.subplot(3, 1, 2)
plt.plot(tvec, theta)
plt.ylabel('theta')
plt.subplot(3, 1, 3)
plt.plot(tvec, y, '.', markersize=2)
plt.ylabel('y')
plt.xlabel('time')
plt.tight_layout()
plt.show()